<a href="https://colab.research.google.com/github/hawa1983/Capstone-Final-Modeling-and-Data/blob/main/dow_ridership_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beyond the Weekday/Weekend Binary
## Day-of-Week Demand Regimes in the Post-Telework Era: Evidence from NYC Subway Turnstile Data

**Author:** Fomba Kassoh  
**Target Journal:** Transportation Research Record (TRR)  
**Date:** 2026

### Project Overview

This notebook implements the full analytical pipeline for the TRR publication examining how telework-driven behavioral change has disrupted the traditional weekday/weekend demand binary in the NYC subway system. The analysis proceeds in five stages:

1. **Data Acquisition** — MTA Turnstile data (2019, 2022, 2024) + built environment variables
2. **Data Processing** — Cleaning, aggregation, and 7-day ridership vector construction
3. **Exploratory Analysis** — Descriptive statistics and day-of-week profile visualization
4. **Cluster Analysis** — K-means typology of station demand regimes
5. **Regression Analysis** — Multinomial logistic regression explaining cluster membership

### Research Objectives
- Empirically characterize 7-day ridership profiles at NYC subway stations across three temporal periods
- Identify K-means demand regime clusters that transcend the weekday/weekend binary
- Explain cluster membership using built environment and land use variables
- Derive scheduling implications for MTA service planning

---
## SECTION 0: Environment Setup
Install all required libraries and configure global settings.

In [1]:
# ==============================================================
# Section 0.1 — Install Dependencies
# ==============================================================
# Run this cell once if working in Google Colab or a fresh environment.
# Comment out if packages are already installed.

!pip install pandas numpy scikit-learn matplotlib seaborn geopandas \
             scipy statsmodels requests tqdm plotly --quiet

In [ ]:
# ==============================================================
# Section 0.2 — Import Libraries
# ==============================================================

from __future__ import annotations

import os
import re
import warnings
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline

import statsmodels.api as sm
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print("All libraries loaded successfully.")

All libraries loaded successfully.


In [9]:
# ==============================================================
# Section 0.2b — Google Colab: Mount Google Drive
# ==============================================================

from pathlib import Path

USE_DRIVE = True        # Set False if running locally

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    OUT_DIR = Path("/content/drive/MyDrive/dow_ridership_paper/outputs")
else:
    OUT_DIR = Path("./outputs")   # local fallback

OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUT_DIR}")
print(f"Drive mounted: {USE_DRIVE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output directory: /content/drive/MyDrive/dow_ridership_paper/outputs
Drive mounted: True


In [10]:
# ==============================================================
# Section 0.3 — Global Configuration
# ==============================================================
# Central config block — modify here, not scattered through the notebook.

import matplotlib.pyplot as plt

# ---- Study periods ----
PERIODS = {
    "precovid": {
        "label": "Pre-COVID Baseline",
        "years": [2017, 2018, 2019],
        "baseline_file": "station_daily_precovid_avg.csv",
    },
    "2022": {
        "label": "Early Recovery",
        "year": 2022,
    },
    "2024": {
        "label": "Stabilized New Normal",
        "year": 2024,
    },
}

# ---- Days of week (Mon=0, Sun=6) ----
DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
DOW_MAP = {
    0: "Mon",
    1: "Tue",
    2: "Wed",
    3: "Thu",
    4: "Fri",
    5: "Sat",
    6: "Sun",
}

# ---- K-means ----
K_RANGE = range(2, 9)   # sweep K=2..8
K_FINAL = 5             # update after elbow/silhouette analysis
KMEANS_SEED = 42
KMEANS_INIT = 10        # number of random initializations

# ---- Catchment radius for built environment join ----
CATCHMENT_MILES = 0.5

# ---- Plot style ----
PALETTE = ["#1F5C99", "#E8702A", "#2E9E4F", "#C03B2B", "#7B4EA0"]

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("Configuration complete.")

Configuration complete.


---
## SECTION 1: Data Acquisition

### 1.1 MTA Turnstile Data

The MTA publishes weekly turnstile data at `data.ny.gov`. Each row represents cumulative entries and exits for a single turnstile unit over a ~4-hour audit interval.

**Key fields:** `C/A` (control area), `UNIT`, `SCP` (subunit/channel/position), `STATION`, `LINENAME`, `DATE`, `TIME`, `ENTRIES`, `EXITS`

**Strategy:** Download all weekly files for calendar years 2019, 2022, and 2024, compute net entries per turnstile per audit period (handling counter resets), aggregate to station-day level, and then to station × day-of-week averages.

**Note on station naming:** The MTA turnstile `STATION` field does not map 1:1 to station complexes. The crosswalk from station name → station complex ID is handled in Section 2.

In [11]:
# ==============================================================
# Replacement Section — Pre-COVID Baseline from Data.NY.gov
# Uses annual archived turnstile datasets instead of old weekly MTA files
# ==============================================================

import time
import datetime
from datetime import date, timedelta
from pathlib import Path

import pandas as pd
import requests
from tqdm import tqdm


# -------------------------------------------------------
# OUTPUT LOCATION
# -------------------------------------------------------

OUT_DIR = Path("/content/drive/MyDrive/dow_ridership_paper/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "raw_turnstile").mkdir(exist_ok=True)

DOW_MAP = {
    0: "Mon",
    1: "Tue",
    2: "Wed",
    3: "Thu",
    4: "Fri",
    5: "Sat",
    6: "Sun",
}

DOW_LABELS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

PRE_COVID_YEARS = [2017, 2018, 2019]

# Official Data.NY.gov annual archived datasets
DATA_NY_DATASET_IDS = {
    2017: "v5y5-mwpb",
    2018: "bjcb-yee3",
    2019: "xfn5-qji9",
}

TURNSTILE_REQUIRED_COLS = {
    "C/A",
    "Unit",
    "SCP",
    "Station",
    "Date",
    "Time",
    "Entries",
}


# -------------------------------------------------------
# DOWNLOAD ANNUAL CSV FROM DATA.NY.GOV
# -------------------------------------------------------

def download_year_csv(year: int) -> Path:
    """
    Download annual archived turnstile CSV from Data.NY.gov.
    Saves one raw file per year:
        raw_turnstile/turnstile_2017.csv
        raw_turnstile/turnstile_2018.csv
        raw_turnstile/turnstile_2019.csv
    """
    dataset_id = DATA_NY_DATASET_IDS[year]
    url = f"https://data.ny.gov/api/views/{dataset_id}/rows.csv?accessType=DOWNLOAD"

    cache_f = OUT_DIR / "raw_turnstile" / f"turnstile_{year}.csv"

    if cache_f.exists() and cache_f.stat().st_size > 1_000_000:
        print(f"  {year}: raw annual CSV already cached: {cache_f}")
        return cache_f

    print(f"\n  {year}: downloading annual CSV from Data.NY.gov...")
    print(f"  URL: {url}")

    headers = {"User-Agent": "Mozilla/5.0"}

    with requests.get(url, headers=headers, stream=True, timeout=120) as r:
        if r.status_code != 200:
            raise RuntimeError(f"{year}: download failed with HTTP {r.status_code}")

        total = int(r.headers.get("content-length", 0))

        with open(cache_f, "wb") as f:
            if total > 0:
                pbar = tqdm(total=total, unit="B", unit_scale=True, desc=f"  {year}")
            else:
                pbar = None

            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    if pbar is not None:
                        pbar.update(len(chunk))

            if pbar is not None:
                pbar.close()

    # Validate columns
    test = pd.read_csv(cache_f, nrows=5, low_memory=False)
    test.columns = test.columns.str.strip()

    missing = TURNSTILE_REQUIRED_COLS - set(test.columns)
    if missing:
        raise RuntimeError(
            f"{year}: downloaded CSV is missing required columns: {missing}\n"
            f"Columns found: {list(test.columns)}"
        )

    print(f"  {year}: download complete and validated.")
    return cache_f


# -------------------------------------------------------
# BUILD STATION-DAILY PER YEAR
# -------------------------------------------------------

def build_station_daily(year: int, chunk_size: int = 750_000) -> pd.DataFrame:
    """
    Build station × date daily entries for one year.

    Uses annual Data.NY.gov CSV instead of weekly MTA text files.
    Handles the large annual files in chunks, then sorts once before diffing.
    """
    cache_out = OUT_DIR / f"station_daily_{year}.csv"

    if cache_out.exists():
        df = pd.read_csv(cache_out, parse_dates=["date"])
        print(
            f"  {year}: loaded station_daily cache — "
            f"{df['station_name'].nunique()} stations | "
            f"{df['date'].nunique()} days | "
            f"{len(df):,} rows"
        )
        return df

    raw_csv = download_year_csv(year)

    print(f"\n  {year}: reading annual CSV in chunks...")

    keep_cols = [
        "C/A",
        "Unit",
        "SCP",
        "Station",
        "Date",
        "Time",
        "Entries",
    ]

    frames = []

    reader = pd.read_csv(
        raw_csv,
        usecols=keep_cols,
        chunksize=chunk_size,
        low_memory=False,
    )

    for chunk in tqdm(reader, desc=f"  {year} chunks"):
        chunk.columns = chunk.columns.str.strip()

        chunk["DATETIME"] = pd.to_datetime(
            chunk["Date"].astype(str).str.strip() + " " +
            chunk["Time"].astype(str).str.strip(),
            format="%m/%d/%Y %H:%M:%S",
            errors="coerce",
        )

        chunk = chunk.dropna(subset=["DATETIME"])
        chunk = chunk[chunk["DATETIME"].dt.year == year].copy()

        if len(chunk) == 0:
            continue

        chunk["Entries"] = pd.to_numeric(chunk["Entries"], errors="coerce")
        chunk = chunk.dropna(subset=["Entries"])

        frames.append(
            chunk[
                [
                    "C/A",
                    "Unit",
                    "SCP",
                    "Station",
                    "DATETIME",
                    "Entries",
                ]
            ]
        )

    if not frames:
        raise RuntimeError(f"{year}: no valid records after reading annual CSV.")

    raw = pd.concat(frames, ignore_index=True)
    print(f"    Raw rows before cleaning: {len(raw):,}")

    # Sort within each physical turnstile
    unit_key = ["C/A", "Unit", "SCP"]

    raw = raw.sort_values(unit_key + ["DATETIME"]).reset_index(drop=True)

    # Compute net entries from cumulative counter
    raw["net_entries"] = (
        raw.groupby(unit_key)["Entries"]
           .diff()
           .clip(lower=0, upper=10_000)
           .fillna(0)
    )

    # Date and DOW
    raw["date"] = raw["DATETIME"].dt.normalize()
    raw["dow"] = raw["DATETIME"].dt.dayofweek
    raw["dow_label"] = raw["dow"].map(DOW_MAP)

    # Exclusion flags from your existing code
    raw["date_str"] = raw["date"].dt.strftime("%Y-%m-%d")
    raw["month_day"] = list(zip(raw["date"].dt.month, raw["date"].dt.day))

    holiday_mask = raw["month_day"].isin(HOLIDAY_MD)
    event_mask = raw["date_str"].isin(EXCLUDE_DATES)

    excluded = int((holiday_mask | event_mask).sum())

    raw = raw[~(holiday_mask | event_mask)].drop(
        columns=["date_str", "month_day"]
    ).copy()

    print(f"    Rows removed by exclusion flags: {excluded:,}")
    print(f"    Rows after exclusions:           {len(raw):,}")

    # Aggregate to station × date
    daily = (
        raw.groupby(["Station", "date", "dow", "dow_label"])["net_entries"]
           .sum()
           .reset_index()
           .rename(columns={"Station": "station_name"})
    )

    daily.to_csv(cache_out, index=False)

    print(
        f"    Final: {daily['station_name'].nunique()} stations | "
        f"{daily['date'].nunique()} days | "
        f"{len(daily):,} rows"
    )

    return daily


# -------------------------------------------------------
# BUILD 3-YEAR PRE-COVID AVERAGE
# -------------------------------------------------------

def build_precovid_average(cleaned: dict) -> pd.DataFrame:
    """
    Build 3-year averaged pre-COVID baseline:
    station × day-of-week mean entries across 2017, 2018, 2019.
    """
    cache_out = OUT_DIR / "station_daily_precovid_avg.csv"

    if cache_out.exists():
        df = pd.read_csv(cache_out)
        print(
            f"\nPre-COVID average: loaded from cache — "
            f"{df['station_name'].nunique()} stations | "
            f"{len(df):,} rows"
        )
        return df

    frames = []

    for yr in PRE_COVID_YEARS:
        if yr not in cleaned:
            print(f"  Warning: {yr} missing from cleaned dict — skipping")
            continue

        dow_agg = (
            cleaned[yr]
            .groupby(["station_name", "dow", "dow_label"])["net_entries"]
            .mean()
            .reset_index()
            .rename(columns={"net_entries": f"mean_entries_{yr}"})
        )

        frames.append(
            dow_agg.set_index(["station_name", "dow", "dow_label"])
        )

    if not frames:
        raise RuntimeError("No pre-COVID years available.")

    combined = frames[0]

    for f in frames[1:]:
        combined = combined.join(f, how="outer")

    combined = combined.reset_index()

    yr_cols = [
        f"mean_entries_{yr}"
        for yr in PRE_COVID_YEARS
        if f"mean_entries_{yr}" in combined.columns
    ]

    combined["mean_entries_precovid"] = combined[yr_cols].mean(
        axis=1,
        skipna=True,
    )

    combined["n_years_available"] = combined[yr_cols].notna().sum(axis=1)

    n_partial = int((combined["n_years_available"] < 3).sum())

    combined.to_csv(cache_out, index=False)

    print("\nPre-COVID 3-year average built:")
    print(f"  Stations:        {combined['station_name'].nunique()}")
    print(f"  DOW rows:        {len(combined):,}")
    print(f"  Full 3-yr rows:  {(combined['n_years_available'] == 3).sum():,}")
    print(f"  Partial rows:    {n_partial:,}")

    print("\nSample Monday high-volume stations:")
    sample = (
        combined[combined["dow_label"] == "Mon"]
        .nlargest(5, "mean_entries_precovid")
        [
            [
                "station_name",
                "dow_label",
                "mean_entries_precovid",
                "n_years_available",
            ]
        ]
    )

    print(sample.to_string(index=False))

    return combined


# -------------------------------------------------------
# RUN
# -------------------------------------------------------

print("\n" + "=" * 55)
print("Building pre-COVID baseline from Data.NY.gov: 2017, 2018, 2019")
print("=" * 55)

cleaned = {}

for yr in PRE_COVID_YEARS:
    cleaned[yr] = build_station_daily(yr)

print("\n" + "=" * 55)
print("Computing 3-year pre-COVID DOW average...")
print("=" * 55)

precovid_avg = build_precovid_average(cleaned)


# -------------------------------------------------------
# SUMMARY
# -------------------------------------------------------

print("\n" + "=" * 55)
print("SUMMARY")
print("=" * 55)

for yr in PRE_COVID_YEARS:
    df = cleaned[yr]

    n_st = df["station_name"].nunique()
    n_day = df["date"].nunique()
    n_rows = len(df)

    dow_counts = df.groupby("dow")["date"].nunique()
    min_days = dow_counts.min()
    max_days = dow_counts.max()

    print(f"\n  cleaned[{yr}]:")
    print(f"    Stations:          {n_st}")
    print(f"    Calendar days:     {n_day}")
    print(f"    Station-day rows:  {n_rows:,}")
    print(f"    DOW day range:     {min_days}–{max_days} days per DOW")

print("\n  precovid_avg:")
print(f"    Stations:     {precovid_avg['station_name'].nunique()}")
print(f"    Total rows:   {len(precovid_avg):,}")

print(f"\nAll files saved to: {OUT_DIR}")


Building pre-COVID baseline from Data.NY.gov: 2017, 2018, 2019
  2017: loaded station_daily cache — 380 stations | 247 days | 92,356 rows
  2018: loaded station_daily cache — 379 stations | 315 days | 117,701 rows
  2019: loaded station_daily cache — 379 stations | 285 days | 107,522 rows

Computing 3-year pre-COVID DOW average...

Pre-COVID average: loaded from cache — 381 stations | 2,662 rows

SUMMARY

  cleaned[2017]:
    Stations:          380
    Calendar days:     247
    Station-day rows:  92,356
    DOW day range:     34–36 days per DOW

  cleaned[2018]:
    Stations:          379
    Calendar days:     315
    Station-day rows:  117,701
    DOW day range:     39–48 days per DOW

  cleaned[2019]:
    Stations:          379
    Calendar days:     285
    Station-day rows:  107,522
    DOW day range:     37–44 days per DOW

  precovid_avg:
    Stations:     381
    Total rows:   2,662

All files saved to: /content/drive/MyDrive/dow_ridership_paper/outputs


### 1.2 MTA Station Complex Crosswalk

The `STATION` field in turnstile data uses MTA-internal names that do not align perfectly with official station complex IDs. We use the MTA's published station complexes file to map turnstile station names → station complex IDs, enabling consistent spatial joins with ACS and land use data.

In [18]:
# ==============================================================
# Section 1.2 — Station Complex Crosswalk
#
# Purpose:
#   Loads the MTA Subway Stations and Complexes file and
#   constructs a crosswalk between turnstile STATION names
#   and official station_complex_id values.
#
# Source: MTA Open Data / data.ny.gov
#   "MTA Subway Stations and Complexes"
#
# Output:
#   station_crosswalk_df: DataFrame with columns
#     [station_name, station_complex_id, complex_name,
#      borough, daytime_routes, complex_latitude, complex_longitude]
# ==============================================================

STATION_COMPLEX_URL = (
    "https://data.ny.gov/api/views/i9wp-a4ja/rows.csv?accessType=DOWNLOAD"
)


def load_station_complexes(url: str) -> pd.DataFrame:
    """
    Download the MTA station complexes file and return a clean DataFrame.
    """
    cache = OUT_DIR / "mta_station_complexes.csv"
    if cache.exists():
        df = pd.read_csv(cache)
    else:
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        df = pd.read_csv(StringIO(resp.text))
        df.to_csv(cache, index=False)

    # Standardize column names to snake_case
    df.columns = (
        df.columns
          .str.strip()
          .str.lower()
          .str.replace(r"[\s/]+", "_", regex=True)
    )
    print(f"Station complexes loaded: {len(df):,} rows")
    print("Columns:", list(df.columns[:10]))
    return df


station_complex_df = load_station_complexes(STATION_COMPLEX_URL)

# ---- Preview ----
station_complex_df.head(3)

Station complexes loaded: 445 rows
Columns: ['complex_id', 'is_complex', 'number_of_stations_in_complex', 'stop_name', 'display_name', 'constituent_station_names', 'station_ids', 'gtfs_stop_ids', 'borough', 'cbd']


,complex_id,is_complex,number_of_stations_in_complex,stop_name,display_name,constituent_station_names,station_ids,gtfs_stop_ids,borough,cbd,daytime_routes,structure_type,latitude,longitude,ada,ada_notes
0,398,False,1,77 St,77 St (6),77 St,398,627,M,False,6,Subway,40.773620,-73.959874,0,NaN
1,399,False,1,68 St-Hunter College,68 St-Hunter College (6),68 St-Hunter College,399,628,M,False,6,Subway,40.768141,-73.963870,1,NaN
2,403,False,1,33 St,33 St (6),33 St,403,632,M,True,6,Subway,40.746081,-73.982076,0,NaN


### 1.3 Built Environment Data

We collect three categories of built environment variables:
- **ACS (Census):** Population density, median household income, vehicle ownership, commute mode share — aggregated to 0.5-mile station buffer
- **LEHD/LODES:** Job counts by sector (office vs. essential vs. retail) within 0.5-mile buffer
- **MTA Metadata:** ADA accessibility, structure type (elevated/underground), borough

In [19]:
# ==============================================================
# Section 1.3a — ACS Data via Census API
#
# Purpose:
#   Fetch block-group level ACS 5-year estimates for NYC.
#
# Output:
#   acs_bg_2019.csv
#   acs_bg_2022.csv
# ==============================================================

import time
import pandas as pd
import requests

# -------------------------------------------------------
# CENSUS API KEY
# -------------------------------------------------------
# Get a free key here:
# https://api.census.gov/data/key_signup.html
#
# Paste your key between the quotes.

CENSUS_API_KEY = "31796ddadb573a901960c3f4b507d61c1924c7f7"


# -------------------------------------------------------
# ACS VARIABLES
# -------------------------------------------------------

ACS_VARS = {
    "B01003_001E": "total_population",
    "B08201_001E": "total_households",
    "B08201_002E": "no_vehicle_hh",
    "B19013_001E": "median_hh_income",
    "B08301_001E": "total_commuters",
    "B08301_010E": "transit_commuters",
}

# NYC FIPS county codes:
# Manhattan = 061
# Bronx = 005
# Brooklyn/Kings = 047
# Queens = 081
# Staten Island/Richmond = 085

NYC_COUNTIES = ["061", "005", "047", "081", "085"]
STATE_FIPS = "36"

# ACS years:
# Use ACS 2019 for pre-COVID baseline.
# Use ACS 2022 for 2022 and 2024 analysis.
ACS_YEAR_MAP = {
    2019: 2019,
    2022: 2022,
    2024: 2022,
}


# -------------------------------------------------------
# FETCH ACS BLOCK GROUPS
# -------------------------------------------------------

def fetch_acs_block_groups(
    acs_year: int,
    state: str,
    counties: list,
    variables: dict,
    api_key: str,
    retries: int = 3,
) -> pd.DataFrame:
    """
    Fetch ACS 5-year block group estimates for NYC counties.

    Returns a DataFrame with:
        GEOID
        NAME
        total_population
        total_households
        no_vehicle_hh
        median_hh_income
        total_commuters
        transit_commuters
    """

    if api_key == "PASTE_YOUR_CENSUS_API_KEY_HERE" or not api_key.strip():
        raise ValueError(
            "You must paste your Census API key into CENSUS_API_KEY before running this cell."
        )

    cache = OUT_DIR / f"acs_bg_{acs_year}.csv"

    if cache.exists():
        print(f"  ACS {acs_year}: loaded from cache")
        return pd.read_csv(cache, dtype={"GEOID": str})

    base_url = f"https://api.census.gov/data/{acs_year}/acs/acs5"

    get_vars = ["NAME", "GEO_ID"] + list(variables.keys())

    frames = []

    for county in counties:
        print(f"  ACS {acs_year}: fetching county {county}...")

        # Use list-of-tuples so repeated "in" parameters are handled correctly.
        # Census block-group hierarchy:
        # state -> county -> tract -> block group
        params = [
            ("get", ",".join(get_vars)),
            ("for", "block group:*"),
            ("in", f"state:{state}"),
            ("in", f"county:{county}"),
            ("in", "tract:*"),
            ("key", api_key),
        ]

        success = False

        for attempt in range(retries):
            try:
                resp = requests.get(base_url, params=params, timeout=60)

                if resp.status_code != 200:
                    print(
                        f"    Attempt {attempt + 1}: county {county} "
                        f"returned HTTP {resp.status_code}"
                    )
                    print("    Response preview:")
                    print(resp.text[:500])
                    time.sleep(2 ** attempt)
                    continue

                try:
                    data = resp.json()
                except Exception as e:
                    print(
                        f"    Attempt {attempt + 1}: county {county} returned non-JSON response."
                    )
                    print(f"    JSON error: {type(e).__name__}: {e}")
                    print("    Response preview:")
                    print(resp.text[:500])
                    time.sleep(2 ** attempt)
                    continue

                if not isinstance(data, list) or len(data) < 2:
                    print(f"    County {county}: no data returned.")
                    print("    Response preview:")
                    print(str(data)[:500])
                    break

                df = pd.DataFrame(data[1:], columns=data[0])
                frames.append(df)

                print(f"    County {county}: {len(df):,} block groups")
                success = True
                break

            except Exception as e:
                print(
                    f"    Attempt {attempt + 1}: county {county} failed: "
                    f"{type(e).__name__}: {e}"
                )
                time.sleep(2 ** attempt)

        if not success:
            print(f"    Warning: county {county} was skipped.")

    if not frames:
        raise RuntimeError(
            f"ACS {acs_year}: no counties returned data. "
            "Check your Census API key, variable names, or internet access."
        )

    combined = pd.concat(frames, ignore_index=True)

    # -------------------------------------------------------
    # Build clean 12-digit block group GEOID
    # -------------------------------------------------------

    combined["GEOID"] = (
        combined["state"].astype(str).str.zfill(2)
        + combined["county"].astype(str).str.zfill(3)
        + combined["tract"].astype(str).str.zfill(6)
        + combined["block group"].astype(str)
    )

    # Rename ACS variable columns
    combined = combined.rename(columns=variables)

    # Convert ACS variables to numeric.
    # Census missing/suppressed values often appear as large negative values.
    for col in variables.values():
        combined[col] = pd.to_numeric(combined[col], errors="coerce")
        combined[col] = combined[col].where(combined[col] >= 0)

    keep_cols = ["GEOID", "NAME"] + list(variables.values())
    combined = combined[keep_cols].copy()

    combined = combined.drop_duplicates(subset=["GEOID"])

    combined.to_csv(cache, index=False)

    print(f"  ACS {acs_year}: saved {len(combined):,} block groups to {cache}")

    return combined


# -------------------------------------------------------
# RUN ACS FETCH
# -------------------------------------------------------

acs_data = {}

for acs_yr in sorted(set(ACS_YEAR_MAP.values())):
    print(f"\nFetching ACS {acs_yr}...")
    acs_data[acs_yr] = fetch_acs_block_groups(
        acs_year=acs_yr,
        state=STATE_FIPS,
        counties=NYC_COUNTIES,
        variables=ACS_VARS,
        api_key=CENSUS_API_KEY,
    )

print("\nACS fetch complete.")

for yr, df in acs_data.items():
    print(f"  ACS {yr}: {len(df):,} block groups")


Fetching ACS 2019...
  ACS 2019: fetching county 061...
    County 061: 1,170 block groups
  ACS 2019: fetching county 005...
    County 005: 1,154 block groups
  ACS 2019: fetching county 047...
    County 047: 2,085 block groups
  ACS 2019: fetching county 081...
    County 081: 1,746 block groups
  ACS 2019: fetching county 085...
    County 085: 338 block groups
  ACS 2019: saved 6,493 block groups to /content/drive/MyDrive/dow_ridership_paper/outputs/acs_bg_2019.csv

Fetching ACS 2022...
  ACS 2022: fetching county 061...
    County 061: 1,292 block groups
  ACS 2022: fetching county 005...
    County 005: 1,182 block groups
  ACS 2022: fetching county 047...
    County 047: 2,156 block groups
  ACS 2022: fetching county 081...
    County 081: 1,803 block groups
  ACS 2022: fetching county 085...
    County 085: 374 block groups
  ACS 2022: saved 6,807 block groups to /content/drive/MyDrive/dow_ridership_paper/outputs/acs_bg_2022.csv

ACS fetch complete.
  ACS 2019: 6,493 block g

In [22]:
# ==============================================================
# Section 1.3b — LEHD/LODES Job Data
# ==============================================================

import pandas as pd
import requests
from io import BytesIO

LODES_BASE = "https://lehd.ces.census.gov/data/lodes/LODES8/ny/wac/"

# 2019 period uses 2019 job data.
# 2022 and 2024 periods use 2021 job data.
LODES_YEARS = {
    2019: 2019,
    2022: 2021,
    2024: 2021,
}

LODES_VARS = {
    "w_geocode": "block_fips",
    "C000": "total_jobs",
    "CNS09": "finance_jobs",
    "CNS12": "professional_jobs",
    "CNS07": "retail_jobs",
    "CNS15": "healthcare_jobs",
    "CNS18": "food_service_jobs",
}


def fetch_lodes_wac(lodes_year: int) -> pd.DataFrame:
    """
    Download and cache LODES WAC file for New York State.

    WAC = Workplace Area Characteristics.
    This gives job counts by workplace location.

    Returns:
        Block-group-level job counts.
    """
    cache = OUT_DIR / f"lodes_wac_{lodes_year}.csv"

    if cache.exists():
        print(f"  LODES {lodes_year}: loaded from cache")
        return pd.read_csv(cache, dtype={"GEOID": str})

    filename = f"ny_wac_S000_JT00_{lodes_year}.csv.gz"
    url = LODES_BASE + filename

    print(f"  Downloading: {url}")

    resp = requests.get(url, timeout=120)
    resp.raise_for_status()

    # Read compressed .csv.gz content correctly
    df = pd.read_csv(
        BytesIO(resp.content),
        compression="gzip",
        dtype={"w_geocode": str},
        low_memory=False,
    )

    # Keep only variables we need
    keep_cols = [c for c in LODES_VARS.keys() if c in df.columns]
    df = df[keep_cols].copy()

    # Rename columns
    df = df.rename(columns=LODES_VARS)

    # Make sure expected job columns exist
    required_after_rename = [
        "block_fips",
        "total_jobs",
        "finance_jobs",
        "professional_jobs",
        "retail_jobs",
        "healthcare_jobs",
        "food_service_jobs",
    ]

    missing = [c for c in required_after_rename if c not in df.columns]

    if missing:
        raise RuntimeError(
            f"LODES {lodes_year}: missing expected columns: {missing}\n"
            f"Columns found: {list(df.columns)}"
        )

    # Convert job columns to numeric
    job_cols = [
        "total_jobs",
        "finance_jobs",
        "professional_jobs",
        "retail_jobs",
        "healthcare_jobs",
        "food_service_jobs",
    ]

    for col in job_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    # Derived fields
    df["office_jobs"] = df["finance_jobs"] + df["professional_jobs"]
    df["essential_jobs"] = df["healthcare_jobs"] + df["food_service_jobs"]

    # Convert 15-digit Census block FIPS to 12-digit block group GEOID
    df["GEOID"] = df["block_fips"].astype(str).str.zfill(15).str[:12]

    # Aggregate block-level jobs to block group
    agg_cols = [
        "total_jobs",
        "finance_jobs",
        "professional_jobs",
        "retail_jobs",
        "healthcare_jobs",
        "food_service_jobs",
        "office_jobs",
        "essential_jobs",
    ]

    bg = (
        df.groupby("GEOID", as_index=False)[agg_cols]
          .sum()
    )

    bg.to_csv(cache, index=False)

    print(f"  LODES {lodes_year}: saved {len(bg):,} block groups to {cache}")

    return bg


# -------------------------------------------------------
# RUN
# -------------------------------------------------------

lodes_data = {}

for lodes_yr in sorted(set(LODES_YEARS.values())):
    print(f"\nFetching LODES {lodes_yr}...")
    lodes_data[lodes_yr] = fetch_lodes_wac(lodes_yr)

print("\nLODES fetch complete.")

for yr, df in lodes_data.items():
    print(f"  LODES {yr}: {len(df):,} block groups")


Fetching LODES 2019...
  Downloading: https://lehd.ces.census.gov/data/lodes/LODES8/ny/wac/ny_wac_S000_JT00_2019.csv.gz
  LODES 2019: saved 15,698 block groups to /content/drive/MyDrive/dow_ridership_paper/outputs/lodes_wac_2019.csv

Fetching LODES 2021...
  Downloading: https://lehd.ces.census.gov/data/lodes/LODES8/ny/wac/ny_wac_S000_JT00_2021.csv.gz
  LODES 2021: saved 15,678 block groups to /content/drive/MyDrive/dow_ridership_paper/outputs/lodes_wac_2021.csv

LODES fetch complete.
  LODES 2019: 15,698 block groups
  LODES 2021: 15,678 block groups


---
## SECTION 2: Data Processing

### 2.1 Turnstile Cleaning and Net Entry Computation

Raw MTA turnstile data records **cumulative** entry/exit counts. Net entries per audit interval must be computed by differencing consecutive readings within each `(C/A, UNIT, SCP)` turnstile unit.

**Key data quality issues:**
- **Counter resets:** Turnstile counters occasionally reset to zero or roll over. Readings that produce negative differences or implausibly large jumps (>10,000 per 4-hour period) are treated as resets and capped at zero.
- **Irregular audit intervals:** Most audits occur every ~4 hours but occasionally skipped. Net entries are normalized to a per-hour rate before aggregation.
- **Station name inconsistencies:** Resolved via the crosswalk loaded in Section 1.2.

In [28]:
# ==============================================================
# FAST Section 2.1 — Post-COVID Station-Complex Daily Ridership
# Monthly server-side aggregation from Data.NY.gov
#
# Output:
#   station_complex_daily_2022.csv
#   station_complex_daily_2024.csv
# ==============================================================

import time
import pandas as pd
import requests
from io import StringIO

HOURLY_RIDERSHIP_DATASET_ID = "wujg-7c2s"
HOURLY_RIDERSHIP_BASE = (
    f"https://data.ny.gov/resource/{HOURLY_RIDERSHIP_DATASET_ID}.csv"
)

POST_COVID_YEARS = [2022, 2023, 2024]


def fetch_one_month_daily(year: int, month: int, retries: int = 4) -> pd.DataFrame:
    """
    Fetch one month of station-complex daily ridership using
    Data.NY.gov server-side aggregation.
    """

    start = pd.Timestamp(year=year, month=month, day=1)

    if month == 12:
        end = pd.Timestamp(year=year + 1, month=1, day=1)
    else:
        end = pd.Timestamp(year=year, month=month + 1, day=1)

    where_clause = (
        f"transit_timestamp >= '{start.strftime('%Y-%m-%dT00:00:00')}' "
        f"AND transit_timestamp < '{end.strftime('%Y-%m-%dT00:00:00')}'"
    )

    select_clause = """
        station_complex_id,
        station_complex,
        borough,
        date_trunc_ymd(transit_timestamp) AS date,
        sum(ridership) AS net_entries,
        sum(transfers) AS transfers
    """

    group_clause = """
        station_complex_id,
        station_complex,
        borough,
        date_trunc_ymd(transit_timestamp)
    """

    params = {
        "$select": select_clause,
        "$where": where_clause,
        "$group": group_clause,
        "$order": "station_complex_id, date",
        "$limit": 50000,
    }

    for attempt in range(retries):
        try:
            resp = requests.get(
                HOURLY_RIDERSHIP_BASE,
                params=params,
                timeout=180,
            )

            if resp.status_code != 200:
                print(f"    {year}-{month:02d}: HTTP {resp.status_code}")
                print(resp.text[:500])
                time.sleep(2 ** attempt)
                continue

            df = pd.read_csv(StringIO(resp.text))

            print(f"    {year}-{month:02d}: {len(df):,} aggregated rows")
            return df

        except requests.exceptions.ReadTimeout:
            print(
                f"    {year}-{month:02d}: timeout on attempt {attempt + 1}; retrying..."
            )
            time.sleep(2 ** attempt)

        except Exception as e:
            print(
                f"    {year}-{month:02d}: failed on attempt {attempt + 1}: "
                f"{type(e).__name__}: {e}"
            )
            time.sleep(2 ** attempt)

    raise RuntimeError(f"{year}-{month:02d}: failed after {retries} attempts")


def fetch_station_complex_daily_by_month(year: int) -> pd.DataFrame:
    """
    Fetch one full year of station-complex daily ridership by downloading
    one month at a time, then combining the months.
    """

    cache = OUT_DIR / f"station_complex_daily_{year}.csv"

    if cache.exists():
        print(f"  {year}: loaded station-complex daily from cache")
        return pd.read_csv(cache, parse_dates=["date"])

    print(f"\n  {year}: downloading daily station-complex ridership by month...")

    frames = []

    for month in range(1, 13):
        month_df = fetch_one_month_daily(year, month)
        frames.append(month_df)

    daily = pd.concat(frames, ignore_index=True)

    daily.columns = (
        daily.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[\s/]+", "_", regex=True)
    )

    # Standardize and clean types
    daily["date"] = pd.to_datetime(daily["date"], errors="coerce")

    daily["net_entries"] = pd.to_numeric(
        daily["net_entries"],
        errors="coerce"
    ).fillna(0)

    if "transfers" in daily.columns:
        daily["transfers"] = pd.to_numeric(
            daily["transfers"],
            errors="coerce"
        ).fillna(0)
    else:
        daily["transfers"] = 0

    # Add day-of-week
    daily["dow"] = daily["date"].dt.dayofweek
    daily["dow_label"] = daily["dow"].map(DOW_MAP)

    # Rename station_complex to complex_name
    if "station_complex" in daily.columns:
        daily = daily.rename(columns={"station_complex": "complex_name"})

    # Remove any accidental duplicate rows after combining months
    group_cols = [
        "station_complex_id",
        "complex_name",
        "borough",
        "date",
        "dow",
        "dow_label",
    ]

    daily = (
        daily
        .groupby(group_cols, as_index=False)
        .agg(
            net_entries=("net_entries", "sum"),
            transfers=("transfers", "sum"),
        )
    )

    daily.to_csv(cache, index=False)

    print(
        f"  {year}: saved {len(daily):,} complex-day rows | "
        f"{daily['station_complex_id'].nunique()} complexes | "
        f"{daily['date'].nunique()} days"
    )

    return daily


# -------------------------------------------------------
# Optional cleanup of old raw hourly cache files
# -------------------------------------------------------

for yr in POST_COVID_YEARS:
    old_cache = OUT_DIR / f"hourly_ridership_{yr}.csv"
    if old_cache.exists():
        old_cache.unlink()
        print(f"Deleted old raw hourly cache: {old_cache}")


# -------------------------------------------------------
# RUN
# -------------------------------------------------------

station_complex_daily = {}

for yr in POST_COVID_YEARS:
    station_complex_daily[yr] = fetch_station_complex_daily_by_month(yr)

print("\nPost-COVID station-complex daily ridership complete.")

for yr, df in station_complex_daily.items():
    print(
        f"  {yr}: {df['station_complex_id'].nunique()} complexes, "
        f"{df['date'].nunique()} days, {len(df):,} rows"
    )


  2022: downloading daily station-complex ridership by month...
    2022-01: timeout on attempt 1; retrying...
    2022-01: timeout on attempt 2; retrying...
    2022-01: timeout on attempt 3; retrying...
    2022-01: timeout on attempt 4; retrying...


RuntimeError: 2022-01: failed after 4 attempts

In [26]:
# # ==============================================================
# # Section 2.1 — Post-COVID Station-Complex Daily Ridership
# # Uses MTA Subway Hourly Ridership: 2020–2024
# #
# # Output:
# #   station_compAlex_daily_2022.csv
# #   station_complex_daily_2024.csv
# # ==============================================================

# import pandas as pd
# import requests
# from pathlib import Path
# from io import StringIO

# HOURLY_RIDERSHIP_DATASET_ID = "wujg-7c2s"

# HOURLY_RIDERSHIP_BASE = (
#     f"https://data.ny.gov/resource/{HOURLY_RIDERSHIP_DATASET_ID}.csv"
# )

# POST_COVID_YEARS = [2022, 2024]


# def fetch_hourly_ridership_year(year: int, limit: int = 50000) -> pd.DataFrame:
#     """
#     Download MTA hourly subway ridership for one calendar year
#     from Data.NY.gov using Socrata paging.

#     Returns raw hourly station-complex-level records.
#     """

#     cache = OUT_DIR / f"hourly_ridership_{year}.csv"

#     if cache.exists():
#         print(f"  {year}: loaded hourly ridership from cache")
#         return pd.read_csv(cache, parse_dates=["transit_timestamp"])

#     print(f"  {year}: downloading hourly station-complex ridership...")

#     frames = []
#     offset = 0

#     # Pull only the requested calendar year.
#     where_clause = (
#         f"transit_timestamp between '{year}-01-01T00:00:00' "
#         f"and '{year + 1}-01-01T00:00:00'"
#     )

#     while True:
#         params = {
#             "$limit": limit,
#             "$offset": offset,
#             "$where": where_clause,
#         }

#         resp = requests.get(
#             HOURLY_RIDERSHIP_BASE,
#             params=params,
#             timeout=120,
#         )

#         if resp.status_code != 200:
#             print("Request URL:")
#             print(resp.url)
#             print("Response preview:")
#             print(resp.text[:500])
#             raise RuntimeError(
#                 f"{year}: Data.NY.gov request failed with HTTP {resp.status_code}"
#             )

#         chunk = pd.read_csv(StringIO(resp.text))

#         if len(chunk) == 0:
#             break

#         frames.append(chunk)

#         print(f"    downloaded {offset + len(chunk):,} rows...")

#         if len(chunk) < limit:
#             break

#         offset += limit

#     if not frames:
#         raise RuntimeError(f"{year}: no hourly ridership records downloaded.")

#     df = pd.concat(frames, ignore_index=True)

#     df.to_csv(cache, index=False)

#     print(f"  {year}: saved {len(df):,} hourly rows to {cache}")

#     return df


# def build_station_complex_daily_from_hourly(year: int) -> pd.DataFrame:
#     """
#     Convert hourly station-complex ridership into daily station-complex totals.
#     """

#     cache = OUT_DIR / f"station_complex_daily_{year}.csv"

#     if cache.exists():
#         print(f"  {year}: loaded station-complex daily from cache")
#         return pd.read_csv(cache, parse_dates=["date"])

#     hourly = fetch_hourly_ridership_year(year)

#     # Standardize column names
#     hourly.columns = (
#         hourly.columns
#         .str.strip()
#         .str.lower()
#         .str.replace(r"[\s/]+", "_", regex=True)
#     )

#     print(f"  {year}: columns found:")
#     print(list(hourly.columns))

#     # Expected key columns in Data.NY.gov hourly ridership file.
#     # The dataset usually includes:
#     # transit_timestamp
#     # station_complex_id
#     # station_complex
#     # borough
#     # payment_method
#     # fare_class_category
#     # ridership
#     # transfers

#     required = {"transit_timestamp", "station_complex_id", "ridership"}

#     missing = required - set(hourly.columns)

#     if missing:
#         raise RuntimeError(
#             f"{year}: missing required columns: {missing}\n"
#             f"Columns found: {list(hourly.columns)}"
#         )

#     hourly["transit_timestamp"] = pd.to_datetime(
#         hourly["transit_timestamp"],
#         errors="coerce",
#     )

#     hourly = hourly.dropna(subset=["transit_timestamp"]).copy()

#     hourly = hourly[hourly["transit_timestamp"].dt.year == year].copy()

#     hourly["ridership"] = pd.to_numeric(
#         hourly["ridership"],
#         errors="coerce",
#     ).fillna(0)

#     hourly["date"] = hourly["transit_timestamp"].dt.normalize()
#     hourly["dow"] = hourly["date"].dt.dayofweek
#     hourly["dow_label"] = hourly["dow"].map(DOW_MAP)

#     # Optional metadata columns if present
#     group_cols = [
#         "station_complex_id",
#         "date",
#         "dow",
#         "dow_label",
#     ]

#     for optional_col in ["station_complex", "borough"]:
#         if optional_col in hourly.columns:
#             group_cols.insert(1, optional_col)

#     daily = (
#         hourly
#         .groupby(group_cols, as_index=False)
#         .agg(
#             net_entries=("ridership", "sum"),
#             transfers=("transfers", "sum") if "transfers" in hourly.columns else ("ridership", "size"),
#         )
#     )

#     # Rename station_complex to complex_name if present
#     if "station_complex" in daily.columns:
#         daily = daily.rename(columns={"station_complex": "complex_name"})

#     daily.to_csv(cache, index=False)

#     print(
#         f"  {year}: {daily['station_complex_id'].nunique()} complexes | "
#         f"{daily['date'].nunique()} days | "
#         f"{len(daily):,} complex-day rows"
#     )

#     return daily


# # -------------------------------------------------------
# # RUN
# # -------------------------------------------------------

# station_complex_daily = {}

# for yr in POST_COVID_YEARS:
#     print(f"\nBuilding station-complex daily ridership for {yr}...")
#     station_complex_daily[yr] = build_station_complex_daily_from_hourly(yr)

# # Load pre-COVID complex baseline if already created later/earlier
# precovid_avg = pd.read_csv(
#     OUT_DIR / "station_daily_precovid_avg.csv"
# )

# print("\nPost-COVID station-complex daily ridership complete.")

# for yr, df in station_complex_daily.items():
#     print(
#         f"  {yr}: {df['station_complex_id'].nunique()} complexes, "
#         f"{df['date'].nunique()} days, {len(df):,} rows"
#     )

# print(
#     f"  Pre-COVID station-level baseline loaded: "
#     f"{precovid_avg['station_name'].nunique()} stations, "
#     f"{len(precovid_avg):,} station-DOW rows"
# )


Building station-complex daily ridership for 2022...
  2022: downloading hourly station-complex ridership...
    downloaded 50,000 rows...
    downloaded 100,000 rows...
    downloaded 150,000 rows...
    downloaded 200,000 rows...
    downloaded 250,000 rows...
    downloaded 300,000 rows...
    downloaded 350,000 rows...
    downloaded 400,000 rows...
    downloaded 450,000 rows...
    downloaded 500,000 rows...
    downloaded 550,000 rows...
    downloaded 600,000 rows...
    downloaded 650,000 rows...
    downloaded 700,000 rows...
    downloaded 750,000 rows...
    downloaded 800,000 rows...
    downloaded 850,000 rows...
    downloaded 900,000 rows...
    downloaded 950,000 rows...
    downloaded 1,000,000 rows...
    downloaded 1,050,000 rows...
    downloaded 1,100,000 rows...
    downloaded 1,150,000 rows...
    downloaded 1,200,000 rows...
    downloaded 1,250,000 rows...
    downloaded 1,300,000 rows...
    downloaded 1,350,000 rows...
    downloaded 1,400,000 rows...
    d

KeyboardInterrupt: 

### 2.2 Station Complex Join

Join cleaned turnstile data to station complex IDs using the MTA crosswalk. Station names in turnstile data are fuzzy-matched to complex names where exact matches fail.

In [ ]:
# ==============================================================
# Section 2.2 — Station Name → Complex ID Join
#
# Purpose:
#   Merges turnstile station names to official station_complex_id
#   using the MTA crosswalk file.
#
# Strategy:
#   1. Exact match on normalized station name.
#   2. Remaining unmatched: fuzzy match using difflib.
#   3. Stations with no plausible match are dropped with a report.
#
# Output:
#   station_daily_with_complex_{year}.csv
# ==============================================================

import difflib


def normalize_name(name: str) -> str:
    """Lowercase, strip punctuation, collapse whitespace."""
    name = str(name).lower()
    name = re.sub(r"[^a-z0-9 ]", " ", name)
    return re.sub(r"\s+", " ", name).strip()


def build_crosswalk(complex_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a name → complex_id crosswalk from the station complex file.
    Includes both the primary complex name and stop_name variants.
    """
    # Identify the correct column names (may vary by download date)
    id_col   = [c for c in complex_df.columns if "complex" in c and "id" in c][0]
    name_col = [c for c in complex_df.columns if "complex" in c and "name" in c][0]
    lat_col  = [c for c in complex_df.columns if "latitude"  in c][0]
    lon_col  = [c for c in complex_df.columns if "longitude" in c][0]

    xwalk = complex_df[[id_col, name_col, lat_col, lon_col]].drop_duplicates()
    xwalk = xwalk.rename(columns={
        id_col:   "station_complex_id",
        name_col: "complex_name",
        lat_col:  "complex_lat",
        lon_col:  "complex_lon",
    })
    xwalk["key"] = xwalk["complex_name"].apply(normalize_name)
    return xwalk


def join_complex(daily_df: pd.DataFrame,
                 xwalk: pd.DataFrame,
                 fuzzy_cutoff: float = 0.75) -> pd.DataFrame:
    """
    Join station_daily to complex crosswalk.
    Returns enriched DataFrame with station_complex_id appended.
    """
    daily_df = daily_df.copy()
    daily_df["key"] = daily_df["station_name"].apply(normalize_name)

    # ---- Exact match ----
    exact_map = xwalk.set_index("key")[
        ["station_complex_id", "complex_name", "complex_lat", "complex_lon"]
    ].to_dict("index")

    unmatched_keys = set(daily_df["key"].unique()) - set(exact_map.keys())

    # ---- Fuzzy match for unmatched ----
    all_xwalk_keys = list(exact_map.keys())
    fuzzy_map = {}
    for ukey in unmatched_keys:
        matches = difflib.get_close_matches(
            ukey, all_xwalk_keys, n=1, cutoff=fuzzy_cutoff
        )
        if matches:
            fuzzy_map[ukey] = exact_map[matches[0]]

    combined_map = {**exact_map, **fuzzy_map}
    still_unmatched = set(daily_df["key"].unique()) - set(combined_map.keys())

    if still_unmatched:
        print(f"  Warning: {len(still_unmatched)} station names unmatched:")
        for s in sorted(still_unmatched)[:10]:
            print(f"    '{s}'")

    daily_df["station_complex_id"] = daily_df["key"].map(
        lambda k: combined_map.get(k, {}).get("station_complex_id")
    )
    daily_df["complex_name"] = daily_df["key"].map(
        lambda k: combined_map.get(k, {}).get("complex_name")
    )
    daily_df["complex_lat"] = daily_df["key"].map(
        lambda k: combined_map.get(k, {}).get("complex_lat")
    )
    daily_df["complex_lon"] = daily_df["key"].map(
        lambda k: combined_map.get(k, {}).get("complex_lon")
    )
    daily_df = daily_df.dropna(subset=["station_complex_id"]).copy()
    daily_df = daily_df.drop(columns=["key"])
    return daily_df


# ---- Build crosswalk and join ----
xwalk_df = build_crosswalk(station_complex_df)

cleaned_with_complex = {}
for yr in [2019, 2022, 2024]:
    cache = OUT_DIR / f"station_daily_with_complex_{yr}.csv"
    if cache.exists():
        cleaned_with_complex[yr] = pd.read_csv(cache, parse_dates=["date"])
        print(f"  {yr}: loaded from cache")
    else:
        print(f"  Joining {yr}...")
        df = join_complex(cleaned[yr], xwalk_df)
        df.to_csv(cache, index=False)
        cleaned_with_complex[yr] = df
        print(f"  {yr}: {df['station_complex_id'].nunique()} complexes retained")

### 2.3 7-Day Ridership Vector Construction

The core analytical unit is a **7-dimensional normalized ridership vector** per station per period.

For each station-complex:
1. Compute the mean daily entries for each of the 7 days of week across the full study year
2. Normalize by the station's total weekly volume (sum across all 7 days = 1.0)

Normalization ensures clustering captures **demand shape** (when people ride) rather than **scale** (how many ride). A large hub station and a small neighborhood station can share the same demand regime.

In [ ]:
# ==============================================================
# Section 2.3 — 7-Day Ridership Vector Construction
#
# Purpose:
#   Converts station-day data to normalized 7-day ridership
#   vectors for use in K-means clustering.
#
# Processing Steps:
#   1. Group by station_complex_id × dow.
#   2. Compute mean daily entries for each day-of-week.
#   3. Pivot to wide format: rows=stations, cols=Mon..Sun.
#   4. Normalize each row by its row-sum (weekly total).
#   5. Drop stations with missing DOW coverage (< 7 days observed).
#   6. Flag stations with < 50 average weekday entries (very low volume).
#
# Output:
#   dow_vectors_{year}.csv  — normalized 7-day vectors
#   dow_raw_{year}.csv      — raw mean entries (for descriptive stats)
# ==============================================================

def build_dow_vectors(daily_df: pd.DataFrame, year: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build normalized and raw day-of-week ridership vectors.
    Returns (normalized_df, raw_df).
    """
    cache_norm = OUT_DIR / f"dow_vectors_{year}.csv"
    cache_raw  = OUT_DIR / f"dow_raw_{year}.csv"

    if cache_norm.exists() and cache_raw.exists():
        print(f"  {year}: loaded DOW vectors from cache")
        return (
            pd.read_csv(cache_norm),
            pd.read_csv(cache_raw),
        )

    # ---- Step 1-2: Mean entries per station × DOW ----
    agg = (
        daily_df
        .groupby(["station_complex_id", "complex_name",
                  "complex_lat", "complex_lon", "dow"])["net_entries"]
        .mean()
        .reset_index()
        .rename(columns={"net_entries": "mean_entries"})
    )

    # ---- Step 3: Pivot to wide format ----
    raw = agg.pivot_table(
        index=["station_complex_id", "complex_name", "complex_lat", "complex_lon"],
        columns="dow",
        values="mean_entries",
    ).reset_index()

    # Rename DOW columns: 0→Mon, 1→Tue, ...
    raw = raw.rename(columns=DOW_MAP)
    dow_cols = DOW_LABELS  # ["Mon", "Tue", ..., "Sun"]

    # ---- Step 5: Drop stations missing any DOW ----
    raw = raw.dropna(subset=dow_cols).copy()

    # ---- Step 4: Normalize ----
    norm = raw.copy()
    row_sums = norm[dow_cols].sum(axis=1)
    norm[dow_cols] = norm[dow_cols].div(row_sums, axis=0)

    # ---- Step 6: Low-volume flag ----
    weekday_cols = ["Mon", "Tue", "Wed", "Thu", "Fri"]
    raw["avg_weekday_entries"]  = raw[weekday_cols].mean(axis=1)
    raw["low_volume_flag"]      = raw["avg_weekday_entries"] < 50
    norm["low_volume_flag"]     = raw["low_volume_flag"].values
    norm["avg_weekday_entries"] = raw["avg_weekday_entries"].values

    raw.to_csv(cache_raw, index=False)
    norm.to_csv(cache_norm, index=False)

    n_low = norm["low_volume_flag"].sum()
    print(f"  {year}: {len(norm)} stations | {n_low} low-volume flagged")
    return norm, raw


dow_vectors = {}  # normalized
dow_raw     = {}  # raw mean entries

for yr in [2019, 2022, 2024]:
    print(f"Building DOW vectors for {yr}...")
    dow_vectors[yr], dow_raw[yr] = build_dow_vectors(cleaned_with_complex[yr], yr)

print("\nDOW vector construction complete.")
dow_vectors[2024].head(3)

---
## SECTION 3: Exploratory Data Analysis

### 3.1 Summary Statistics Table

Table 1 in the paper — mean daily entries by day of week across the three study periods, system-wide.

In [ ]:
# ==============================================================
# Section 3.1 — Summary Statistics: Day-of-Week × Period
#
# Purpose:
#   Computes system-wide mean daily entries by day of week
#   for each study period, and calculates % change from 2019.
#   This becomes Table 1 in the TRR paper.
#
# Output:
#   table1_summary_stats.csv
# ==============================================================

summary_rows = []
for yr in [2019, 2022, 2024]:
    df = dow_raw[yr]
    for day in DOW_LABELS:
        summary_rows.append({
            "period": yr,
            "period_label": PERIODS[str(yr)]["label"],
            "day_of_week": day,
            "mean_entries": df[day].mean(),
            "median_entries": df[day].median(),
            "std_entries": df[day].std(),
        })

summary_df = pd.DataFrame(summary_rows)

# Compute % change vs 2019
baseline = summary_df[summary_df["period"] == 2019].set_index("day_of_week")["mean_entries"]
summary_df["pct_change_vs_2019"] = summary_df.apply(
    lambda r: (r["mean_entries"] - baseline[r["day_of_week"]]) / baseline[r["day_of_week"]] * 100,
    axis=1,
)

# ---- Print formatted Table 1 ----
table1 = summary_df.pivot_table(
    index="day_of_week",
    columns="period_label",
    values=["mean_entries", "pct_change_vs_2019"],
).round(1)

# Preserve DOW order
table1 = table1.reindex(DOW_LABELS)

print("Table 1: System-Wide Mean Daily Entries by Day of Week (Station-Level Average)")
print(table1.to_string())

summary_df.to_csv(OUT_DIR / "table1_summary_stats.csv", index=False)

In [ ]:
# ==============================================================
# Section 3.2 — Figure 2: Overlaid 7-Day Profiles (2019 vs 2024)
#
# Purpose:
#   Visualizes the structural shift in day-of-week ridership
#   profiles between 2019 and 2024 for two contrasting
#   station types:
#     (a) High office-density Manhattan stations
#     (b) Outer-borough residential stations
#
# This becomes Figure 2 in the TRR paper.
# ==============================================================

# Representative stations (update with actual complex IDs after data review)
OFFICE_STATIONS = [
    "Grand Central-42 St",
    "Times Sq-42 St",
    "47-50 Sts-Rockefeller Ctr",
    "34 St-Herald Sq",
    "Fulton St",
]
RESIDENTIAL_STATIONS = [
    "Flatbush Av-Brooklyn College",
    "Pelham Bay Park",
    "Jamaica-179 St",
    "Bay Ridge-95 St",
    "Norwood-205 St",
]


def get_profile(year: int, station_names: list[str]) -> pd.Series:
    """
    Get normalized mean DOW profile across a set of station names.
    """
    df = dow_raw[year]
    mask = df["complex_name"].isin(station_names)
    subset = df[mask][DOW_LABELS]
    if subset.empty:
        print(f"  Warning: no matching stations for {station_names[:2]}...")
        return pd.Series(index=DOW_LABELS, dtype=float)
    profile = subset.mean()
    return profile / profile.sum()  # normalize shape


fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

for ax, names, title in zip(
    axes,
    [OFFICE_STATIONS, RESIDENTIAL_STATIONS],
    ["(a) High Office-Density Stations", "(b) Outer-Borough Residential Stations"],
):
    p2019 = get_profile(2019, names)
    p2024 = get_profile(2024, names)

    x = range(len(DOW_LABELS))
    ax.plot(x, p2019, marker="o", linewidth=2, color=PALETTE[0],
            label="2019 (Pre-COVID Baseline)")
    ax.plot(x, p2024, marker="s", linewidth=2, color=PALETTE[1], linestyle="--",
            label="2024 (Stabilized New Normal)")

    ax.set_xticks(list(x))
    ax.set_xticklabels(DOW_LABELS)
    ax.set_title(title, fontsize=11, fontweight="bold", pad=10)
    ax.set_ylabel("Share of Weekly Ridership", fontsize=9)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.axvspan(0, 0.5, alpha=0.05, color="gray")   # shade Mon
    ax.axvspan(3.5, 4.5, alpha=0.05, color="gray") # shade Fri
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle(
    "Figure 2: Day-of-Week Ridership Profile Shift, 2019 vs. 2024",
    fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig(OUT_DIR / "figure2_dow_profile_shift.png",
            dpi=300, bbox_inches="tight")
plt.show()
print("Figure 2 saved.")

In [ ]:
# ==============================================================
# Section 3.3 — Monday/Friday vs. Tuesday–Thursday Recovery Gap
#
# Purpose:
#   Quantifies the "Mon/Fri gap" — the structural divergence
#   between Mon/Fri recovery and Tue–Thu recovery, the
#   empirical foundation of the paper's core argument.
#
# Metric: Recovery Rate = (2024 mean entries) / (2019 mean entries)
# ==============================================================

recovery_rows = []
for day in DOW_LABELS:
    mean_2019 = dow_raw[2019][day].mean()
    mean_2022 = dow_raw[2022][day].mean()
    mean_2024 = dow_raw[2024][day].mean()
    recovery_rows.append({
        "day": day,
        "recovery_2022": mean_2022 / mean_2019 if mean_2019 > 0 else np.nan,
        "recovery_2024": mean_2024 / mean_2019 if mean_2019 > 0 else np.nan,
        "mean_2019": mean_2019,
        "mean_2024": mean_2024,
    })

recovery_df = pd.DataFrame(recovery_rows).set_index("day").reindex(DOW_LABELS)

# ---- Print recovery table ----
print("Recovery Rates by Day of Week (relative to 2019 baseline)")
print("-" * 55)
for day, row in recovery_df.iterrows():
    bar_2022 = "█" * int(row["recovery_2022"] * 20)
    bar_2024 = "█" * int(row["recovery_2024"] * 20)
    print(f"{day:4s}  2022: {row['recovery_2022']:5.1%} {bar_2022}")
    print(f"      2024: {row['recovery_2024']:5.1%} {bar_2024}")
    print()

# ---- Compute and report the Mon/Fri gap ----
monfri_2024 = recovery_df.loc[["Mon", "Fri"], "recovery_2024"].mean()
tuethu_2024 = recovery_df.loc[["Tue", "Wed", "Thu"], "recovery_2024"].mean()
gap = tuethu_2024 - monfri_2024

print(f"\nMon/Fri average recovery (2024):    {monfri_2024:.1%}")
print(f"Tue–Thu average recovery (2024):    {tuethu_2024:.1%}")
print(f"Mid-week premium over Mon/Fri:      {gap:.1%}  ← KEY FINDING")

recovery_df.to_csv(OUT_DIR / "recovery_rates_by_dow.csv")

---
## SECTION 4: Cluster Analysis

### 4.1 K Selection: Elbow Method + Silhouette Score

We sweep K from 2 to 8. The optimal K is selected at the inflection point of the inertia curve (elbow method), validated by the silhouette score which measures cluster cohesion relative to separation.

In [ ]:
# ==============================================================
# Section 4.1 — K-Selection: Elbow + Silhouette
#
# Purpose:
#   Determines optimal number of clusters K for K-means
#   applied to the normalized 7-day DOW ridership vectors.
#
# Method:
#   - Sweep K = 2..8
#   - Record inertia (within-cluster sum of squares) for elbow method
#   - Record silhouette score for each K
#   - Run K_FINAL = 5 as default; update after inspection
#
# Analysis Period: 2024 (Stabilized New Normal) — primary clustering target
# ==============================================================

CLUSTER_YEAR = 2024  # Primary analysis year

# ---- Prepare feature matrix ----
# Exclude low-volume stations from clustering (noisy profiles)
cluster_df = dow_vectors[CLUSTER_YEAR].copy()
cluster_df = cluster_df[~cluster_df["low_volume_flag"]].copy()

X = cluster_df[DOW_LABELS].values  # shape: (n_stations, 7)
print(f"Feature matrix shape: {X.shape}")
print(f"Stations included: {len(cluster_df)}")

# ---- Standardize ----
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ---- K sweep ----
inertias    = []
silhouettes = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=KMEANS_INIT,
                random_state=KMEANS_SEED, max_iter=500)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels)
    silhouettes.append(sil)
    print(f"  K={k:2d}  inertia={km.inertia_:10.1f}  silhouette={sil:.4f}")

# ---- Plot ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(list(K_RANGE), inertias, "o-", color=PALETTE[0], linewidth=2)
ax1.axvline(K_FINAL, color=PALETTE[1], linestyle="--",
            label=f"Selected K={K_FINAL}")
ax1.set_xlabel("Number of Clusters (K)")
ax1.set_ylabel("Inertia (WCSS)")
ax1.set_title("Elbow Method", fontweight="bold")
ax1.legend()

ax2.plot(list(K_RANGE), silhouettes, "s-", color=PALETTE[2], linewidth=2)
ax2.axvline(K_FINAL, color=PALETTE[1], linestyle="--",
            label=f"Selected K={K_FINAL}")
ax2.set_xlabel("Number of Clusters (K)")
ax2.set_ylabel("Silhouette Score")
ax2.set_title("Silhouette Score", fontweight="bold")
ax2.legend()

plt.suptitle("Figure 1: K-Means Cluster Selection Diagnostics (2024 Data)",
             fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "figure1_k_selection.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"\nSelected K = {K_FINAL} (update K_FINAL in config if diagnostics suggest otherwise)")

In [ ]:
# ==============================================================
# Section 4.2 — Final K-Means Fit
#
# Purpose:
#   Fits K-means with K=K_FINAL on normalized 7-day vectors
#   for all three study periods.
#
# Important: Scaler is fit ONLY on training data (2024 primary).
#   2019 and 2022 vectors are transformed using the same scaler
#   to enable cross-period cluster assignment comparison.
#
# Output:
#   cluster_assignments_{year}.csv — station → cluster label
# ==============================================================

# ---- Fit final K-means on 2024 data ----
km_final = KMeans(
    n_clusters=K_FINAL,
    n_init=KMEANS_INIT,
    random_state=KMEANS_SEED,
    max_iter=1000,
)
km_final.fit(X_scaled)

cluster_assignments = {}

for yr in [2019, 2022, 2024]:
    cache = OUT_DIR / f"cluster_assignments_{yr}.csv"
    if cache.exists():
        cluster_assignments[yr] = pd.read_csv(cache)
        print(f"  {yr}: loaded from cache")
        continue

    # Use only non-low-volume stations present in 2024 clustering universe
    df_yr = dow_vectors[yr].copy()
    df_yr = df_yr[~df_yr["low_volume_flag"]].copy()

    X_yr = df_yr[DOW_LABELS].values
    # Transform using the scaler fitted on 2024
    X_yr_scaled = scaler.transform(X_yr)

    df_yr["cluster"] = km_final.predict(X_yr_scaled)
    df_yr["period"]  = yr

    df_yr.to_csv(cache, index=False)
    cluster_assignments[yr] = df_yr
    print(f"  {yr}: cluster assignments saved")
    print(f"    Distribution: {df_yr['cluster'].value_counts().sort_index().to_dict()}")

print("\nK-means fitting complete.")

In [ ]:
# ==============================================================
# Section 4.3 — Cluster Profile Analysis and Labeling
#
# Purpose:
#   Computes mean normalized DOW profile for each cluster
#   and generates interpretive labels based on profile shape.
#
# This produces Table 2 in the TRR paper.
#
# Cluster label logic (update after inspecting actual profiles):
#   - Sharp Tue–Thu peak + Mon/Fri near weekend levels → Hybrid Office Core
#   - Flat across all 7 days, stable → Essential/Residential Stable
#   - Weekend dominant, moderate weekday → Mixed-Use/Amenity
#   - Elevated all days, modest mid-week → Transit Hub
#   - Weekend > Mon/Fri → Emerging Weekend
# ==============================================================

# ---- Compute cluster centroids in original (non-scaled) normalized space ----
df_2024_clusters = cluster_assignments[2024].copy()

cluster_profiles = (
    df_2024_clusters
    .groupby("cluster")[DOW_LABELS]
    .mean()
    .round(4)
)

print("Table 2: Cluster Profiles — Mean Normalized Day-of-Week Ridership Share (2024)")
print(cluster_profiles.to_string())

# ---- Cluster size ----
cluster_sizes = df_2024_clusters["cluster"].value_counts().sort_index()
print("\nCluster sizes:")
print(cluster_sizes)

# ---- Assign interpretive labels ----
# NOTE: Update these labels based on actual profile inspection
CLUSTER_LABELS = {
    0: "Hybrid Office Core",
    1: "Essential/Residential Stable",
    2: "Mixed-Use / Amenity",
    3: "Transit Hub",
    4: "Emerging Weekend",
}

for yr in [2019, 2022, 2024]:
    cluster_assignments[yr]["cluster_label"] = (
        cluster_assignments[yr]["cluster"].map(CLUSTER_LABELS)
    )

cluster_profiles["cluster_label"] = cluster_profiles.index.map(CLUSTER_LABELS)
cluster_profiles.to_csv(OUT_DIR / "table2_cluster_profiles.csv")

# ---- Print representative stations per cluster ----
print("\nRepresentative Stations by Cluster (top 5 by avg weekday volume):")
for c_id, c_label in CLUSTER_LABELS.items():
    members = (
        df_2024_clusters[df_2024_clusters["cluster"] == c_id]
        .nlargest(5, "avg_weekday_entries")[["complex_name", "avg_weekday_entries"]]
    )
    print(f"\n  Cluster {c_id}: {c_label}")
    print(members.to_string(index=False))

In [ ]:
# ==============================================================
# Section 4.4 — Figure 3: Radar Chart of Cluster Profiles
#
# Purpose:
#   Visualizes the 7-day ridership profile for each cluster
#   as a radar/spider chart.
#   This becomes Figure 3 in the TRR paper.
# ==============================================================

import numpy as np

categories = DOW_LABELS
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the loop

fig, axes = plt.subplots(
    1, K_FINAL,
    figsize=(4 * K_FINAL, 4),
    subplot_kw=dict(polar=True)
)

for idx, (c_id, c_label) in enumerate(CLUSTER_LABELS.items()):
    ax = axes[idx]
    values = cluster_profiles.loc[c_id, DOW_LABELS].tolist()
    values += values[:1]  # close the loop

    ax.plot(angles, values, linewidth=2, color=PALETTE[idx])
    ax.fill(angles, values, alpha=0.20, color=PALETTE[idx])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=8)
    ax.set_yticklabels([])
    ax.set_title(
        f"Cluster {c_id}\n{c_label}",
        fontsize=9, fontweight="bold", pad=15
    )
    n_stations = cluster_sizes.get(c_id, 0)
    ax.text(0, 0, f"n={n_stations}", ha="center", va="center",
            fontsize=8, color="gray")

fig.suptitle(
    "Figure 3: Day-of-Week Demand Regime Profiles by Cluster (2024)",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig(OUT_DIR / "figure3_radar_cluster_profiles.png",
            dpi=300, bbox_inches="tight")
plt.show()
print("Figure 3 saved.")

In [ ]:
# ==============================================================
# Section 4.5 — Cluster Transition Analysis (2019 → 2024)
#
# Purpose:
#   Tracks how stations migrated between demand regime clusters
#   from the pre-COVID baseline to the stabilized new normal.
#   Produces the transition matrix (Table 3 in the TRR paper).
#
# Method:
#   Inner join stations present in both 2019 and 2024;
#   cross-tabulate cluster assignments.
# ==============================================================

# ---- Merge 2019 and 2024 cluster assignments on station_complex_id ----
df_2019 = cluster_assignments[2019][["station_complex_id", "complex_name",
                                      "cluster", "cluster_label"]].copy()
df_2019 = df_2019.rename(columns={"cluster": "cluster_2019",
                                    "cluster_label": "label_2019"})

df_2024 = cluster_assignments[2024][["station_complex_id", "cluster",
                                      "cluster_label"]].copy()
df_2024 = df_2024.rename(columns={"cluster": "cluster_2024",
                                    "cluster_label": "label_2024"})

transitions = df_2019.merge(df_2024, on="station_complex_id", how="inner")

# ---- Transition matrix ----
transition_matrix = pd.crosstab(
    transitions["label_2019"],
    transitions["label_2024"],
    margins=True,
    margins_name="Total",
)

print("Table 3: Cluster Transition Matrix (2019 → 2024)")
print("Rows = 2019 cluster | Columns = 2024 cluster")
print(transition_matrix.to_string())

# ---- Stability metrics ----
n_stable = (transitions["cluster_2019"] == transitions["cluster_2024"]).sum()
n_total  = len(transitions)
pct_stable = n_stable / n_total * 100
print(f"\nStations in same cluster in both periods: {n_stable} / {n_total} ({pct_stable:.1f}%)")
print(f"Stations that migrated to a different cluster: {n_total - n_stable} ({100 - pct_stable:.1f}%)")

# ---- Identify biggest movers (most dramatic migrations) ----
migrants = transitions[
    transitions["cluster_2019"] != transitions["cluster_2024"]
].copy()

# Merge back with volume data to find high-volume migrants
migrants = migrants.merge(
    dow_raw[2024][["station_complex_id", "avg_weekday_entries"]],
    on="station_complex_id",
    how="left"
)
top_migrants = migrants.nlargest(10, "avg_weekday_entries")[
    ["complex_name", "label_2019", "label_2024", "avg_weekday_entries"]
]
print("\nTop 10 High-Volume Stations That Migrated Between Clusters:")
print(top_migrants.to_string(index=False))

transition_matrix.to_csv(OUT_DIR / "table3_transition_matrix.csv")
transitions.to_csv(OUT_DIR / "station_transitions.csv", index=False)

---
## SECTION 5: Regression Analysis

### 5.1 Built Environment Feature Engineering

Spatially join ACS block-group data and LODES job counts to each station's 0.5-mile catchment area. Aggregate demographic and employment variables to the station complex level.

In [ ]:
# ==============================================================
# Section 5.1 — Spatial Join: Built Environment → Stations
#
# Purpose:
#   Aggregates ACS and LODES data to station-level features
#   using 0.5-mile station buffers.
#
# Requires: geopandas, shapely
#
# Processing Steps:
#   1. Project station coordinates to NYC local CRS (EPSG:2263).
#   2. Buffer each station by 0.5 miles.
#   3. Spatially join Census block group geometries to buffers.
#   4. Compute area-weighted mean for population-based variables.
#   5. Sum job counts directly (jobs located within buffer).
#   6. Derive engineered features.
#
# Output:
#   station_features.csv
# ==============================================================

import geopandas as gpd
from shapely.geometry import Point

# ---- Build station GeoDataFrame ----
station_geo_df = cluster_assignments[2024][[
    "station_complex_id", "complex_name",
    "complex_lat", "complex_lon"
]].drop_duplicates().dropna(subset=["complex_lat", "complex_lon"])

station_gdf = gpd.GeoDataFrame(
    station_geo_df,
    geometry=gpd.points_from_xy(
        station_geo_df["complex_lon"],
        station_geo_df["complex_lat"]
    ),
    crs="EPSG:4326",
)

# Project to NYC State Plane (feet) for accurate buffering
station_gdf = station_gdf.to_crs("EPSG:2263")

# 0.5 mile in feet = 2640 feet
BUFFER_FT = CATCHMENT_MILES * 5280
station_buffers = station_gdf.copy()
station_buffers["geometry"] = station_gdf.geometry.buffer(BUFFER_FT)

print(f"Station buffers created: {len(station_buffers)} stations @ {CATCHMENT_MILES}-mile radius")

# ---- Load NYC Census block group geometries (TIGER/Line) ----
TIGER_URL = (
    "https://www2.census.gov/geo/tiger/TIGER2022/BG/tl_2022_36_bg.zip"
)
tiger_cache = OUT_DIR / "tl_2022_36_bg.zip"

if not tiger_cache.exists():
    print("Downloading NYC TIGER block group file...")
    resp = requests.get(TIGER_URL, timeout=120)
    resp.raise_for_status()
    tiger_cache.write_bytes(resp.content)

bg_gdf = gpd.read_file(f"zip://{tiger_cache}")
# Filter to NYC counties
bg_gdf = bg_gdf[
    (bg_gdf["STATEFP"] == "36") &
    (bg_gdf["COUNTYFP"].isin(["061", "005", "047", "081", "085"]))
].copy()
bg_gdf = bg_gdf.to_crs("EPSG:2263")
bg_gdf["GEOID"] = bg_gdf["GEOID"].astype(str).str.zfill(12)

print(f"NYC block groups loaded: {len(bg_gdf)}")

In [ ]:
# ==============================================================
# Section 5.2 — Feature Assembly
#
# Purpose:
#   Merges ACS + LODES data onto block group geometries,
#   performs spatial join to station buffers, and
#   aggregates to station-level features.
#
# Engineered features:
#   pct_no_vehicle   — % households with no vehicle
#   pct_transit_commute — % workers commuting by transit
#   office_job_density  — office jobs per acre in buffer
#   retail_job_density  — retail jobs per acre in buffer
#   essential_job_density — essential jobs per acre in buffer
#   log_office_jobs     — log1p(office_jobs) for skew correction
#   log_total_jobs      — log1p(total_jobs)
#   log_pop_density     — log1p(pop per acre)
#   log_median_income   — log1p(median_hh_income)
#
# Output:
#   station_features.csv
# ==============================================================

features_cache = OUT_DIR / "station_features.csv"

if features_cache.exists():
    station_features = pd.read_csv(features_cache)
    print("Station features loaded from cache.")
else:
    # ---- Merge ACS onto block groups ----
    acs_2022 = acs_data[2022].copy()
    lodes_2021 = lodes_data[2021].copy()

    bg_enriched = bg_gdf.merge(acs_2022, on="GEOID", how="left")
    bg_enriched = bg_enriched.merge(lodes_2021, on="GEOID", how="left")

    # ---- Spatial join: buffers ∩ block groups ----
    joined = gpd.sjoin(
        station_buffers[["station_complex_id", "geometry"]],
        bg_enriched,
        how="left",
        predicate="intersects"
    )

    # Compute buffer area for density calculation
    station_buffers["buffer_area_acres"] = (
        station_buffers.geometry.area / 43560  # sq ft → acres
    )

    # ---- Aggregate to station level ----
    agg = joined.groupby("station_complex_id").agg(
        total_population    = ("total_population",    "sum"),
        total_households    = ("total_households",    "sum"),
        no_vehicle_hh       = ("no_vehicle_hh",       "sum"),
        total_commuters     = ("total_commuters",     "sum"),
        transit_commuters   = ("transit_commuters",   "sum"),
        median_hh_income    = ("median_hh_income",    "median"),
        total_jobs          = ("total_jobs",          "sum"),
        office_jobs         = ("office_jobs",         "sum"),
        retail_jobs         = ("retail_jobs",         "sum"),
        essential_jobs      = ("essential_jobs",      "sum"),
    ).reset_index()

    # Merge buffer area
    agg = agg.merge(
        station_buffers[["station_complex_id", "buffer_area_acres"]],
        on="station_complex_id"
    )

    # ---- Derived features ----
    agg["pct_no_vehicle"] = (
        agg["no_vehicle_hh"] / agg["total_households"].replace(0, np.nan)
    )
    agg["pct_transit_commute"] = (
        agg["transit_commuters"] / agg["total_commuters"].replace(0, np.nan)
    )
    agg["pop_density"]           = agg["total_population"] / agg["buffer_area_acres"]
    agg["office_job_density"]    = agg["office_jobs"]  / agg["buffer_area_acres"]
    agg["retail_job_density"]    = agg["retail_jobs"]  / agg["buffer_area_acres"]
    agg["essential_job_density"] = agg["essential_jobs"] / agg["buffer_area_acres"]

    # Log transformations for skewed variables
    agg["log_office_jobs"]    = np.log1p(agg["office_jobs"])
    agg["log_total_jobs"]     = np.log1p(agg["total_jobs"])
    agg["log_pop_density"]    = np.log1p(agg["pop_density"])
    agg["log_median_income"]  = np.log1p(agg["median_hh_income"])

    agg.to_csv(features_cache, index=False)
    station_features = agg
    print(f"Station features assembled: {len(station_features)} stations")

print(station_features.describe().round(2).to_string())

In [ ]:
# ==============================================================
# Section 5.3 — Regression Model: Multinomial Logistic
#
# Purpose:
#   Explains cluster membership using built environment
#   features via multinomial logistic regression.
#
# Dependent variable: cluster label (0–K_FINAL-1), 2024 assignment
# Independent variables: log-transformed built environment features
# Reference category: Cluster 1 (Essential/Residential Stable)
#
# Estimation: statsmodels MNLogit for full coefficient table
#             with p-values and standard errors.
# Validation: 5-fold stratified cross-validation (sklearn)
#
# Output:
#   table4_regression_results.csv
# ==============================================================

# ---- Assemble regression dataset ----
reg_df = cluster_assignments[2024][[
    "station_complex_id", "complex_name", "cluster", "cluster_label"
]].merge(
    station_features,
    on="station_complex_id",
    how="inner"
)

print(f"Regression dataset: {len(reg_df)} stations")
print(f"Cluster distribution:\n{reg_df['cluster'].value_counts().sort_index()}")

# ---- Feature selection ----
FEATURE_COLS = [
    "log_office_jobs",
    "log_total_jobs",
    "log_pop_density",
    "log_median_income",
    "pct_no_vehicle",
    "pct_transit_commute",
    "retail_job_density",
    "essential_job_density",
]

# Drop rows with any missing feature
reg_df = reg_df.dropna(subset=FEATURE_COLS).copy()
print(f"After dropping NAs: {len(reg_df)} stations")

X_reg = reg_df[FEATURE_COLS].copy()
y_reg = reg_df["cluster"].copy()

# ---- Fit Multinomial Logit via statsmodels (for full output) ----
X_sm = sm.add_constant(X_reg)
mnlogit = sm.MNLogit(y_reg, X_sm)
result = mnlogit.fit(method="bfgs", maxiter=500, disp=False)

print("\nMultinomial Logit Results Summary:")
print(result.summary())

In [ ]:
# ==============================================================
# Section 5.4 — Regression Output Table (Table 4)
#
# Purpose:
#   Formats the multinomial logit coefficient table for
#   publication in TRR (Table 4).
#   Includes: coefficients, standard errors, p-values,
#   significance stars, and pseudo-R².
#
# Also runs 5-fold cross-validation for accuracy reporting.
# ==============================================================


def sig_stars(p: float) -> str:
    """Return significance stars for p-value."""
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    if p < 0.10:  return "†"
    return ""


# ---- Extract coefficient table ----
coef_tables = []

for outcome_idx in range(1, K_FINAL):  # MNLogit omits reference category (0)
    outcome_label = CLUSTER_LABELS.get(outcome_idx, f"Cluster {outcome_idx}")
    coefs  = result.params.iloc[:, outcome_idx - 1]
    pvals  = result.pvalues.iloc[:, outcome_idx - 1]
    bse    = result.bse.iloc[:, outcome_idx - 1]

    tbl = pd.DataFrame({
        "outcome":    outcome_label,
        "variable":   coefs.index,
        "coef":       coefs.values,
        "std_err":    bse.values,
        "p_value":    pvals.values,
        "stars":      [sig_stars(p) for p in pvals.values],
    })
    coef_tables.append(tbl)

all_coefs = pd.concat(coef_tables, ignore_index=True)
all_coefs["coef_str"] = (
    all_coefs["coef"].map("{:.3f}".format) +
    all_coefs["stars"]
)
all_coefs["se_str"] = "(" + all_coefs["std_err"].map("{:.3f}".format) + ")"

print("Table 4: Multinomial Logit Coefficients")
print(f"Reference category: {CLUSTER_LABELS[0]}")
print(f"Pseudo-R² (McFadden): {result.prsquared:.4f}")
print(f"Log-likelihood: {result.llf:.2f}")
print(f"N = {len(reg_df)}")
print()

pivot = all_coefs.pivot_table(
    index="variable", columns="outcome",
    values="coef_str", aggfunc="first"
)
print(pivot.to_string())

all_coefs.to_csv(OUT_DIR / "table4_regression_results.csv", index=False)

# ---- 5-Fold Cross-Validation (sklearn) ----
print("\nCross-Validation (5-fold, Stratified):")
lr = LogisticRegression(
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=1000,
    C=1.0,
    random_state=KMEANS_SEED,
)
cv_pipe = Pipeline([("scaler", StandardScaler()), ("clf", lr)])
cv_scores = cross_val_score(
    cv_pipe, X_reg.values, y_reg.values,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=KMEANS_SEED),
    scoring="accuracy",
)
print(f"  Accuracy per fold: {[f'{s:.3f}' for s in cv_scores]}")
print(f"  Mean accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

---
## SECTION 6: Visualization for Publication

### 6.1 Figure 4: NYC Map — Stations Colored by Cluster Assignment

In [ ]:
# ==============================================================
# Section 6.1 — Figure 4: NYC Cluster Map
#
# Purpose:
#   Plots all subway station complexes on a NYC basemap,
#   colored by their 2024 cluster assignment.
#   This becomes Figure 4 in the TRR paper.
#
# Note: Uses geopandas for geometry and matplotlib for rendering.
#   For journal submission, export at 300 DPI as PNG or PDF.
# ==============================================================

# ---- Build GeoDataFrame for plotting ----
map_df = cluster_assignments[2024][[
    "station_complex_id", "complex_name",
    "complex_lat", "complex_lon",
    "cluster", "cluster_label",
]].dropna(subset=["complex_lat", "complex_lon"]).copy()

map_gdf = gpd.GeoDataFrame(
    map_df,
    geometry=gpd.points_from_xy(map_df["complex_lon"], map_df["complex_lat"]),
    crs="EPSG:4326"
)

# ---- NYC Borough boundaries (public domain) ----
NYC_BOROUGHS_URL = (
    "https://data.cityofnewyork.us/api/geospatial/7t3b-ywvw"
    "?method=export&type=GeoJSON"
)
boroughs_cache = OUT_DIR / "nyc_boroughs.geojson"
if not boroughs_cache.exists():
    resp = requests.get(NYC_BOROUGHS_URL, timeout=60)
    boroughs_cache.write_bytes(resp.content)

boroughs_gdf = gpd.read_file(boroughs_cache).to_crs("EPSG:4326")

# ---- Plot ----
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

boroughs_gdf.plot(
    ax=ax,
    color="#F5F5F0",
    edgecolor="#CCCCCC",
    linewidth=0.8,
)

for c_id, c_label in CLUSTER_LABELS.items():
    subset = map_gdf[map_gdf["cluster"] == c_id]
    subset.plot(
        ax=ax,
        color=PALETTE[c_id],
        markersize=18,
        alpha=0.85,
        label=f"Cluster {c_id}: {c_label} (n={len(subset)})",
        zorder=3,
    )

ax.set_axis_off()
ax.legend(
    loc="lower left",
    fontsize=9,
    framealpha=0.9,
    title="Day-of-Week Demand Regime",
    title_fontsize=9,
)
ax.set_title(
    "Figure 4: NYC Subway Station Demand Regime Clusters (2024)",
    fontsize=13, fontweight="bold", pad=15,
)

plt.tight_layout()
plt.savefig(OUT_DIR / "figure4_nyc_cluster_map.png",
            dpi=300, bbox_inches="tight")
plt.show()
print("Figure 4 saved.")

---
## SECTION 7: Robustness Checks

### 7.1 K Sensitivity — Re-run Clustering at K=3, 4, 6

TRR reviewers will ask whether results are sensitive to K selection. We rerun clustering at K±1 and K+1, check if major findings hold, and report silhouette scores.

In [ ]:
# ==============================================================
# Section 7.1 — Robustness: K Sensitivity Analysis
#
# Purpose:
#   Re-runs K-means at K = {K_FINAL-2, K_FINAL-1, K_FINAL+1}
#   and verifies that the Hybrid Office Core cluster
#   (sharp Tue–Thu peak with Mon/Fri collapse) persists
#   across all K specifications.
#
# Key robustness test: At any K, is there ALWAYS a cluster
#   where Mon + Fri share < (Tue + Wed + Thu share) / 3?
# ==============================================================

K_ROBUSTNESS = [K_FINAL - 2, K_FINAL - 1, K_FINAL + 1]
K_ROBUSTNESS = [k for k in K_ROBUSTNESS if k >= 2]  # ensure K >= 2

print("Robustness Check: K Sensitivity")
print("=" * 50)

for k_test in K_ROBUSTNESS:
    km_test = KMeans(
        n_clusters=k_test,
        n_init=KMEANS_INIT,
        random_state=KMEANS_SEED,
        max_iter=500,
    )
    labels_test = km_test.fit_predict(X_scaled)
    sil_test = silhouette_score(X_scaled, labels_test)

    # Compute centroids in normalized space
    test_df = cluster_df.copy()
    test_df["cluster_test"] = labels_test
    centroids_test = test_df.groupby("cluster_test")[DOW_LABELS].mean()

    # Check: does an office-core cluster exist?
    centroids_test["monfri_share"] = centroids_test[["Mon", "Fri"]].sum(axis=1)
    centroids_test["tuethu_share"] = centroids_test[["Tue", "Wed", "Thu"]].sum(axis=1)
    centroids_test["is_office_core"] = (
        centroids_test["tuethu_share"] > centroids_test["monfri_share"] * 1.15
    )

    n_office_core = centroids_test["is_office_core"].sum()
    print(f"\nK={k_test}: silhouette={sil_test:.4f} | "
          f"Office-core clusters detected: {n_office_core}")
    print(centroids_test[["Mon", "Tue", "Wed", "Thu", "Fri",
                           "monfri_share", "tuethu_share",
                           "is_office_core"]].round(3).to_string())

print("\nRobustness check complete.")

In [ ]:
# ==============================================================
# Section 7.2 — Robustness: Weekend Definition Sensitivity
#
# Purpose:
#   Tests whether conclusions change if Monday and Friday
#   are reclassified as "soft weekend" days and the analysis
#   is re-run with only Tue–Thu as "core weekdays".
#
# This directly supports the paper's argument that the
# weekday/weekend binary is no longer empirically valid.
# ==============================================================

# ---- Binary schedule test ----
# Under current MTA scheduling, Tue–Thu get the same
# headway/service as Mon and Fri.
# We test: is demand on Mon and Fri significantly different
# from Sat and Sun at Hybrid Office Core stations?

# Get Hybrid Office Core stations
office_core_ids = cluster_assignments[2024][
    cluster_assignments[2024]["cluster_label"] == "Hybrid Office Core"
]["station_complex_id"].unique()

office_raw_2024 = dow_raw[2024][
    dow_raw[2024]["station_complex_id"].isin(office_core_ids)
].copy()

# Paired t-test: Mon vs Sat at office-core stations
for day_a, day_b, description in [
    ("Mon", "Sat", "Mon vs Sat (office-core, 2024)"),
    ("Fri", "Sat", "Fri vs Sat (office-core, 2024)"),
    ("Tue", "Mon", "Tue vs Mon (office-core, 2024) — mid-week premium"),
]:
    t_stat, p_val = stats.ttest_rel(
        office_raw_2024[day_a].dropna(),
        office_raw_2024[day_b].dropna(),
    )
    direction = "higher" if t_stat > 0 else "lower"
    print(f"{description}")
    print(f"  t = {t_stat:.3f}, p = {p_val:.4f}  "
          f"({day_a} is {direction} than {day_b})")
    print()

---
## SECTION 8: Export and Paper Deliverables

Compiles all tables and figures into the final output folder for TRR submission.

In [ ]:
# ==============================================================
# Section 8.1 — Export Final Deliverables Summary
#
# Purpose:
#   Lists all outputs generated by this notebook and
#   confirms they are present in ./outputs/
# ==============================================================

DELIVERABLES = {
    "Tables": {
        "Table 1 — Summary Statistics": "table1_summary_stats.csv",
        "Table 2 — Cluster Profiles": "table2_cluster_profiles.csv",
        "Table 3 — Transition Matrix": "table3_transition_matrix.csv",
        "Table 4 — Regression Results": "table4_regression_results.csv",
    },
    "Figures": {
        "Figure 1 — K Selection Diagnostics": "figure1_k_selection.png",
        "Figure 2 — DOW Profile Shift 2019 vs 2024": "figure2_dow_profile_shift.png",
        "Figure 3 — Radar Chart Cluster Profiles": "figure3_radar_cluster_profiles.png",
        "Figure 4 — NYC Cluster Map": "figure4_nyc_cluster_map.png",
    },
    "Data": {
        "DOW Vectors 2019": "dow_vectors_2019.csv",
        "DOW Vectors 2022": "dow_vectors_2022.csv",
        "DOW Vectors 2024": "dow_vectors_2024.csv",
        "Cluster Assignments 2019": "cluster_assignments_2019.csv",
        "Cluster Assignments 2022": "cluster_assignments_2022.csv",
        "Cluster Assignments 2024": "cluster_assignments_2024.csv",
        "Station Features": "station_features.csv",
        "Station Transitions 2019→2024": "station_transitions.csv",
        "Recovery Rates by DOW": "recovery_rates_by_dow.csv",
    },
}

print("=" * 60)
print("DELIVERABLES CHECKLIST")
print("=" * 60)

all_present = True
for category, items in DELIVERABLES.items():
    print(f"\n{category}:")
    for label, filename in items.items():
        path = OUT_DIR / filename
        status = "✓" if path.exists() else "✗ MISSING"
        if not path.exists():
            all_present = False
        size = f"({path.stat().st_size:,} bytes)" if path.exists() else ""
        print(f"  [{status}] {label:45s} {size}")

print("\n" + "=" * 60)
if all_present:
    print("All deliverables present. Ready for TRR submission.")
else:
    print("Some deliverables are missing. Re-run relevant sections.")
print("=" * 60)

In [ ]:
# ==============================================================
# Section 8.2 — Key Findings Summary (For Paper Writing)
#
# Purpose:
#   Prints a structured summary of the key quantitative
#   findings for direct use in writing Results and
#   Discussion sections of the TRR paper.
# ==============================================================

print("=" * 60)
print("KEY FINDINGS SUMMARY FOR PAPER WRITING")
print("=" * 60)

print("\n[1] Day-of-Week Recovery Gap (2024 vs 2019):")
for day in DOW_LABELS:
    r = recovery_df.loc[day, "recovery_2024"]
    print(f"  {day}: {r:.1%}")

print(f"\n  Mon/Fri average recovery:  {monfri_2024:.1%}")
print(f"  Tue–Thu average recovery:  {tuethu_2024:.1%}")
print(f"  Mid-week premium:          {gap:.1%}")

print(f"\n[2] Cluster Analysis (K={K_FINAL}, 2024):")
for c_id, c_label in CLUSTER_LABELS.items():
    n = cluster_sizes.get(c_id, 0)
    pct = n / cluster_sizes.sum() * 100
    print(f"  Cluster {c_id} ({c_label}): {n} stations ({pct:.1f}%)")

print(f"\n[3] Cluster Transition (2019 → 2024):")
print(f"  Stations in same cluster:  {n_stable} / {n_total} ({pct_stable:.1f}%)")
print(f"  Stations that migrated:    {n_total - n_stable} ({100 - pct_stable:.1f}%)")

print(f"\n[4] Regression (Multinomial Logit):")
print(f"  Pseudo-R² (McFadden):      {result.prsquared:.4f}")
print(f"  5-fold CV accuracy:        {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"  N (stations):              {len(reg_df)}")

print("\n" + "=" * 60)
print("Notebook execution complete.")
print("=" * 60)